In [1]:
import joblib
import json
import shap
import pandas as pd

In [24]:
model = joblib.load('churn_model_xgb_calibrated.pkl')
with open('churn_model_metadata.json') as f:
    meta = json.load(f)

In [25]:
best_xgb_pipeline = joblib.load('xgb_pipeline.pkl')

In [26]:
xgb_preprocessor = best_xgb_pipeline.named_steps['preprocessor']
xgb_model = best_xgb_pipeline.named_steps['model']

In [27]:
explainer = shap.TreeExplainer(xgb_model)

In [31]:
BINARY_COLS = [
    "Partner", "Dependents", "PhoneService", "MultipleLines", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "PaperlessBilling"
]

FEATURE_LABELS = {
    'ordinal__Contract': "Month-to-Month Contract",
    'scaler__tenure': "Short Tenure",
    'scaler__MonthlyCharges': "High Monthly Charges",
    'scaler__TotalCharges': "Low Total Spend (New Customer)",
    'nominal__InternetService_Fiber optic': "Fiber Optic Internet",
    'nominal__PaymentMethod_Electronic check': "Electronic Check Payment",
    'remainder__OnlineSecurity': "No Online Security",
    'remainder__TechSupport': "No Tech Support",
    'remainder__PaperlessBilling': "Paperless Billing",
}

SUGGESTIONS = {
    'ordinal__Contract': "Offer a discount to switch from month-to-month to a 1-year or 2-year contract.",
    'nominal__InternetService_Fiber optic': "Review fiber pricing or bundle offers — fiber customers show higher churn.",
    'scaler__tenure': "New customer — enroll in an early-engagement or onboarding loyalty program.",
    'scaler__MonthlyCharges': "Consider a loyalty discount or bundle to reduce bill amount.",
    'scaler__TotalCharges': "Low lifetime spend — engage with a value-added service trial.",
    'nominal__PaymentMethod_Electronic check': "Encourage switching to auto-pay (credit card/bank debit) with a small incentive.",
    'remainder__OnlineSecurity': "Offer a free trial of Online Security add-on.",
    'remainder__TechSupport': "Offer a free trial of Tech Support add-on.",
    'remainder__PaperlessBilling': "No action needed — minor factor.",
}

In [32]:
def get_risk_level(proba):
    if proba <= 0.20:
        return "Very Low", "🟢"
    elif proba <= 0.40:
        return "Low", "🟢"
    elif proba <= 0.60:
        return "Moderate", "🟡"
    elif proba <= 0.80:
        return "High", "🟠"
    else:
        return "Critical", "🔴"


In [33]:
def get_priority_stars(risk_level):
    mapping = {"Very Low": 1, "Low": 2, "Moderate": 3, "High": 4, "Critical": 5}
    n = mapping.get(risk_level, 1)
    return "⭐" * n

In [57]:
def explain_prediction(raw_customer_dict):
    customer_df = pd.DataFrame([raw_customer_dict])

    customer_df_encoded = customer_df.copy()
    for col in BINARY_COLS:
        if col in customer_df_encoded.columns and customer_df_encoded[col].dtype == object:
            customer_df_encoded[col] = customer_df_encoded[col].replace({'Yes': 1, 'No': 0})

    customer_df_encoded = customer_df_encoded[meta['feature_columns']]

    proba = model.predict_proba(customer_df_encoded)[:, 1][0]
    pred = int(proba >= meta['chosen_threshold'])
    risk_level, risk_emoji = get_risk_level(proba)

    feature_names = xgb_preprocessor.get_feature_names_out()
    transformed = xgb_preprocessor.transform(customer_df_encoded)
    shap_vals = explainer.shap_values(transformed)[0]

    contrib = pd.Series(shap_vals, index=feature_names).sort_values(ascending=False)
    top_risk_factors = contrib[contrib > 0].head(3)
    top_protective_factors = contrib[contrib < 0].sort_values().head(3)  # most negative first

    readable_risk = [FEATURE_LABELS.get(f, f.split('__')[-1]) for f in top_risk_factors.index]
    readable_protective = [FEATURE_LABELS.get(f, f.split('__')[-1]) for f in top_protective_factors.index]

    suggestions = [SUGGESTIONS.get(f, "Review this factor manually.") for f in top_risk_factors.index]

    print("=" * 52)
    print("         CUSTOMER CHURN PREDICTION REPORT")
    print("=" * 52)
    print(f"Prediction        : {'⚠️ Will Churn' if pred else '✅ Will Stay'}")
    print(f"Probability       : {proba:.1%}")
    print(f"Risk Level        : {risk_emoji} {risk_level}")

    if readable_risk:
        print("\nTop Risk Factors")
        print("-" * 16)
        for i, factor in enumerate(readable_risk, 1):
            print(f"{i}. {factor}")
    else:
        print("\nTop Risk Factors")
        print("-" * 16)
        print("None — no significant churn-driving factors found for this customer.")

    if readable_protective:
        print("\nTop Factors Keeping This Customer Loyal")
        print("-" * 40)
        for i, factor in enumerate(readable_protective, 1):
            print(f"{i}. {factor}")

    print("\nBusiness Interpretation")
    print("-" * 23)
    if readable_risk:
        factor_text = ", ".join(readable_risk[:-1]) + (
            f", and {readable_risk[-1]}" if len(readable_risk) > 1 else readable_risk[0]
        )
        print(f"The customer has a {risk_level.lower()} probability of churning, "
              f"primarily driven by: {factor_text}.")
    else:
        protective_text = ", ".join(readable_protective[:-1]) + (
            f", and {readable_protective[-1]}" if len(readable_protective) > 1 else readable_protective[0]
        )
        print(f"The customer has a {risk_level.lower()} probability of churning. "
              f"No churn-risk factors were identified — instead, the customer is strongly retained by: "
              f"{protective_text}.")

    print("\nRecommended Actions")
    print("-" * 19)
    if suggestions:
        for s in suggestions:
            print(f"✓ {s}")
    else:
        print("✓ No retention action needed — continue standard engagement and loyalty rewards.")

    print("\nPriority")
    print("-" * 8)
    stars = get_priority_stars(risk_level)
    action_text = "Immediate Action Required" if risk_level == "Critical" else \
                  "Prompt Follow-up Recommended" if risk_level == "High" else \
                  "Monitor Periodically" if risk_level == "Moderate" else \
                  "No Immediate Action Needed"
    print(f"{stars} {action_text}")
    print("=" * 52)

In [59]:
new_customer = {
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 2,
    'PhoneService': 'Yes',
    'MultipleLines': 'Yes',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 95.50,
    'TotalCharges': 191.00
}

explain_prediction(new_customer)

         CUSTOMER CHURN PREDICTION REPORT
Prediction        : ⚠️ Will Churn
Probability       : 86.5%
Risk Level        : 🔴 Critical

Top Risk Factors
----------------
1. Month-to-Month Contract
2. Short Tenure
3. Fiber Optic Internet

Top Factors Keeping This Customer Loyal
----------------------------------------
1. SeniorCitizen
2. PhoneService
3. DeviceProtection

Business Interpretation
-----------------------
The customer has a critical probability of churning, primarily driven by: Month-to-Month Contract, Short Tenure, and Fiber Optic Internet.

Recommended Actions
-------------------
✓ Offer a discount to switch from month-to-month to a 1-year or 2-year contract.
✓ New customer — enroll in an early-engagement or onboarding loyalty program.
✓ Review fiber pricing or bundle offers — fiber customers show higher churn.

Priority
--------
⭐⭐⭐⭐⭐ Immediate Action Required


C:\Users\DELL\AppData\Local\Temp\ipykernel_15600\3242577576.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  customer_df_encoded[col] = customer_df_encoded[col].replace({'Yes': 1, 'No': 0})
